# Blender smoke v3 — light_vehicle (self-contained)

Phase 1 G1: 20 кадрів GAZ Tigr через BlenderProc на Colab T4.

**Self-contained:** engine код + config пишуться в Drive з heredoc у цьому notebook —
без Drive FUSE cache lag, без file IDs, без MCP create_file fragility.

**Geometry:** `distance_m` = 3D line-of-sight камера→техніка (200-1000 м).
View angles 10-30° (cruising oblique drone).


In [ ]:
# 1. Mount Drive + paths + mkdir (КРИТИЧНО: створюємо dirs ДО writes)
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE   = '/content/drive/MyDrive/yolo_bb'
REPO    = f'{DRIVE}/datasetforge'
ENGINE  = f'{REPO}/engine'
CONFIGS = f'{REPO}/configs'
ASSETS  = f'{REPO}/assets'
OUT     = f'{REPO}/output/v1_smoke'

for p in (ENGINE, CONFIGS, ASSETS, OUT,
          f'{ASSETS}/hdri', f'{ASSETS}/textures/ground', f'{ASSETS}/models/light_vehicle'):
    pathlib.Path(p).mkdir(parents=True, exist_ok=True)
print('paths ready:', REPO)


In [ ]:
# 2. Install BlenderProc + helpers
!pip install -q blenderproc h5py pyyaml


In [ ]:
# 3. Write engine code from heredoc -> Drive
# (Self-contained: avoid Drive FUSE cache lag at MCP create_file.
#  Source of truth = local datasetforge/engine/*.py у repo.)
import pathlib

RENDER_RUNNER_PY = r'''import blenderproc as bproc  # MUST be first line per BlenderProc CLI check.

# Per-config render orchestrator. Runs only via `blenderproc run <this_file>`.

import argparse
import json
import shutil
import sys
from pathlib import Path

# Add repo root до sys.path щоб `datasetforge.engine.*` імпортувалось.
_REPO_ROOT = Path(__file__).resolve().parent.parent.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import yaml


def parse_args(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", type=Path, required=True)
    ap.add_argument("--n", type=int, default=20, help="кількість кадрів")
    ap.add_argument("--out", type=Path, required=True, help="вихідна папка")
    ap.add_argument("--assets-root", type=Path, default=Path("datasetforge/assets"))
    ap.add_argument("--seed", type=int, default=42)
    return ap.parse_args(argv)


def main(argv=None):
    args = parse_args(argv)

    # bproc вже імпортований на верху файла (BlenderProc requirement).
    # Інші — relative import з datasetforge.engine.
    from datasetforge.engine.scene_builder import CameraSpec, SceneRequest, build_scene
    from datasetforge.engine.bbox_extractor import coco_to_yolo, write_yolo_label
    from datasetforge.engine.camera_sampler import build_grid, sample_round_robin
    from datasetforge.engine.season_lighting import pick_season_assets

    cfg = yaml.safe_load(args.config.read_text(encoding="utf-8"))
    cls_meta = cfg["class"]
    img_w, img_h = cfg["image_size"]
    cam_cfg = cfg["camera"]
    scene_cfg = cfg["scene"]

    bproc.init()

    # GPU enable (Cycles CUDA / OptiX). Без цього BlenderProc дефолтно CPU.
    try:
        import bpy
        prefs = bpy.context.preferences.addons["cycles"].preferences
        # OPTIX краще на NVIDIA RTX, CUDA — на T4/V100/A100/L4.
        for backend in ("OPTIX", "CUDA"):
            try:
                prefs.compute_device_type = backend
                prefs.get_devices()
                for d in prefs.devices:
                    d.use = d.type != "CPU"
                bpy.context.scene.cycles.device = "GPU"
                print(f"[gpu] Cycles backend={backend}, devices={[d.name for d in prefs.devices if d.use]}")
                break
            except Exception as e:
                print(f"[gpu] {backend} failed: {e}")
        else:
            print("[gpu] не вдалось увімкнути GPU, лишаємось на CPU")
    except Exception as e:
        print(f"[gpu] GPU setup error (продовжуємо CPU): {e}")

    grid = build_grid(cam_cfg["distance_m"], cam_cfg["view_angle_deg"])
    samples = sample_round_robin(grid, args.n)

    out_dir = args.out
    img_dir = out_dir / "images"
    lbl_dir = out_dir / "labels"
    meta_dir = out_dir / "metadata"
    for d in (img_dir, lbl_dir, meta_dir):
        d.mkdir(parents=True, exist_ok=True)

    models_dir = args.assets_root / "models" / cls_meta["name"]
    model_variants = []
    for ext in ("*.blend", "*.glb", "*.gltf", "*.fbx", "*.obj"):
        model_variants.extend(models_dir.glob(ext))
    model_variants = sorted(model_variants)
    if not model_variants:
        print(f"[err] no .blend/.glb/.gltf/.fbx/.obj in {models_dir}", file=sys.stderr)
        return 2
    print(f"[models] {len(model_variants)} variants: {[m.name for m in model_variants]}")

    for i, cam_sample in enumerate(samples):
        seed = args.seed + i
        # Round-robin season + landscape + model variant
        season = scene_cfg["seasons"][i % len(scene_cfg["seasons"])]
        landscape = scene_cfg["landscapes"][i % len(scene_cfg["landscapes"])]
        model_path = model_variants[i % len(model_variants)]
        assets = pick_season_assets(args.assets_root, season, seed=seed)

        camera = CameraSpec(
            distance_m=cam_sample.distance_m,
            view_angle_deg=cam_sample.view_angle_deg,
            hfov_deg=cam_cfg["hfov_deg"],
        )
        req = SceneRequest(
            class_name=cls_meta["name"],
            class_id=cls_meta["id"],
            model_path=model_path,
            hdri_path=assets.hdri,
            ground_texture_path=assets.ground_texture,
            camera=camera,
            season=season,
            landscape=landscape,
            weather=scene_cfg["weather"][i % len(scene_cfg["weather"])],
            image_w=img_w,
            image_h=img_h,
            seed=seed,
        )

        # Свіжа сцена кожного кадру (BlenderProc rebuild)
        bproc.utility.reset_keyframes()
        build_scene(req)

        bproc.renderer.set_max_amount_of_samples(64)
        bproc.renderer.enable_segmentation_output(map_by=["category_id", "instance"])
        data = bproc.renderer.render()

        # Тимчасова COCO папка, потім конвертуємо у YOLO
        tmp_coco = out_dir / f"_coco_{i:05d}"
        tmp_coco.mkdir(exist_ok=True)
        bproc.writer.write_coco_annotations(
            str(tmp_coco),
            instance_segmaps=data["instance_segmaps"],
            instance_attribute_maps=data["instance_attribute_maps"],
            colors=data["colors"],
            color_file_format="JPEG",
            jpg_quality=85,
            label_mapping=bproc.utility.LabelIdMapping.from_dict({cls_meta["name"]: cls_meta["id"]}),
        )

        # COCO → YOLO label
        for img_filename, boxes in coco_to_yolo(
            tmp_coco / "coco_annotations.json",
            image_w=img_w, image_h=img_h, min_side_px=10,
        ):
            stem = f"{cls_meta['name']}_{i:05d}"
            # Перенести JPEG у канонічну out_dir/images. img_filename з COCO може бути
            # "images/0.jpg" або "0.jpg" — пробуємо recursive lookup.
            dst_img = img_dir / f"{stem}.jpg"
            src_img = tmp_coco / img_filename
            if not src_img.exists():
                # Fallback: знайти будь-який .jpg всередині tmp_coco
                candidates = list(tmp_coco.rglob("*.jpg")) + list(tmp_coco.rglob("*.jpeg")) + list(tmp_coco.rglob("*.png"))
                if candidates:
                    src_img = candidates[0]
            if src_img.exists():
                shutil.move(str(src_img), dst_img)
            else:
                print(f"[warn] no image source for frame {i}; checked {tmp_coco}", file=sys.stderr)
            write_yolo_label(boxes, lbl_dir / f"{stem}.txt")

            # Metadata sidecar
            import math as _m
            _elev_rad = _m.radians(max(cam_sample.view_angle_deg, 5.0))
            meta = {
                "image_id": stem,
                "source": "synthetic_blender",
                "dataset_version": cfg.get("version", "df-v1.0.0-dev"),
                "seed": seed,
                "distance_m": cam_sample.distance_m,
                "altitude_m": cam_sample.distance_m * _m.sin(_elev_rad),
                "horizontal_distance_m": cam_sample.distance_m * _m.cos(_elev_rad),
                "view_angle_deg": cam_sample.view_angle_deg,
                "hfov_deg": cam_cfg["hfov_deg"],
                "sensor_res": [img_w, img_h],
                "modality": "EO",
                "season": season,
                "landscape": landscape,
                "weather": req.weather,
                "time_of_day": "day",
                "class_name": cls_meta["name"],
                "class_id": cls_meta["id"],
                "n_boxes": len(boxes),
                "has_targets": len(boxes) > 0,
                "model_variant": model_path.name,
                "hdri": assets.hdri.name,
                "ground_texture": assets.ground_texture.name,
            }
            (meta_dir / f"{stem}.json").write_text(
                json.dumps(meta, indent=2, ensure_ascii=False),
                encoding="utf-8",
            )

        shutil.rmtree(tmp_coco, ignore_errors=True)
        _alt = cam_sample.distance_m * _m.sin(_m.radians(max(cam_sample.view_angle_deg, 5.0)))
        print(f"[{i+1}/{args.n}] d={cam_sample.distance_m:.0f}m angle={cam_sample.view_angle_deg:.0f}° "
              f"alt={_alt:.0f}m season={season} model={model_path.name}")

    print(f"[done] {args.n} frames → {out_dir}")
    return 0


if __name__ == "__main__":
    sys.exit(main())
'''

SCENE_BUILDER_PY = r'''"""Blender scene builder через BlenderProc.

Запускається через `blenderproc run datasetforge/engine/render_runner.py ...`
(BlenderProc сам стартує Blender 4.x з потрібним bpy в process).

Importable локально без bpy — `bproc` import зроблений lazy всередині build_scene.
Це дозволяє юніт-тестам тестувати `CameraSpec`/`SceneRequest` дataclasses без Blender.
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from pathlib import Path


@dataclass
class CameraSpec:
    distance_m: float        # 3D line-of-sight camera → vehicle (200-1000 m)
    view_angle_deg: float    # елевація над горизонтом (10-30° drone cruising)
    hfov_deg: float
    sensor_width_mm: float = 35.0

    @property
    def focal_mm(self) -> float:
        return self.sensor_width_mm / (2 * math.tan(math.radians(self.hfov_deg) / 2))


@dataclass
class SceneRequest:
    class_name: str
    class_id: int
    model_path: Path
    hdri_path: Path
    ground_texture_path: Path
    camera: CameraSpec
    season: str
    landscape: str
    weather: str
    image_w: int
    image_h: int
    seed: int


def build_scene(req: SceneRequest):
    """Будує одну сцену під рендер. Повертає (camera_pose_4x4, vehicle_mesh_objs).

    Кроки:
      1. Завантажити vehicle (.blend / .glb / .gltf / .obj / .fbx).
      2. Tag category_id + normalize scale до 5м max-dim + random Z rotation.
      3. Ground plane 10×10км з seasonal PBR-текстурою.
      4. HDRI world background (strength=2.0) + explicit SUN light (energy=5).
      5. Camera pose: distance_m + view_angle_deg → H=d·sin(θ), xy=d·cos(θ).
      6. Camera intrinsics (focal_mm з hfov_deg, sensor_width).
    """
    # Lazy import: bproc + bpy доступні тільки коли запущено через `blenderproc run`.
    import bpy
    import blenderproc as bproc
    import numpy as np

    # 0. CLEANUP previous frame.
    # bproc.utility.reset_keyframes() у render_runner чистить ТІЛЬКИ анімаційні keyframes,
    # mesh+light об'єкти з попереднього кадру лишаються у сцені і накопичуються.
    # Без цього cleanup після 3-х кадрів у сцені 3 машини = "зліплені 3 моделі" артефакт.
    for obj in list(bpy.data.objects):
        if obj.type in ("MESH", "LIGHT"):
            bpy.data.objects.remove(obj, do_unlink=True)
    # Orphan data cleanup щоб memory не роздувалась за 20+ frames (texture/mesh blocks).
    for collection in (bpy.data.meshes, bpy.data.materials,
                       bpy.data.images, bpy.data.lights):
        for item in list(collection):
            if not item.users:
                collection.remove(item)

    # 1. Завантажити vehicle (dispatch за extension)
    ext = req.model_path.suffix.lower()
    if ext == ".blend":
        objs = bproc.loader.load_blend(str(req.model_path))
    elif ext in (".glb", ".gltf"):
        before = set(bpy.data.objects.keys())
        bpy.ops.import_scene.gltf(filepath=str(req.model_path))
        new_names = set(bpy.data.objects.keys()) - before
        objs = [bproc.types.MeshObject(bpy.data.objects[n]) for n in new_names
                if bpy.data.objects[n].type == "MESH"]
    elif ext == ".obj":
        objs = bproc.loader.load_obj(str(req.model_path))
    elif ext == ".fbx":
        before = set(bpy.data.objects.keys())
        bpy.ops.import_scene.fbx(filepath=str(req.model_path))
        new_names = set(bpy.data.objects.keys()) - before
        objs = [bproc.types.MeshObject(bpy.data.objects[n]) for n in new_names
                if bpy.data.objects[n].type == "MESH"]
    else:
        raise RuntimeError(f"unsupported model format: {ext} ({req.model_path})")
    # objs з усіх loader-ів — вже MeshObject; додатковий filter не потрібен.
    vehicle_meshes = [o for o in objs if isinstance(o, bproc.types.MeshObject)]
    if not vehicle_meshes:
        raise RuntimeError(f"no MESH objects in {req.model_path}")
    for o in vehicle_meshes:
        o.set_cp("category_id", req.class_id)

    # Normalize vehicle scale: max-dim → ~5 м.
    # Захист від GLB-файлів з non-meters units (cm/dm/mm), що тягне камеру
    # всередину моделі і дає "50-100 см рендер" замість 200-1000 м.
    TARGET_MAX_DIM_M = 5.0
    combined_bbox = np.vstack([np.array(o.get_bound_box(local_coords=False))
                               for o in vehicle_meshes])
    dims = combined_bbox.max(axis=0) - combined_bbox.min(axis=0)
    current_max = float(dims.max())
    if current_max > 0.01:
        scale_factor = TARGET_MAX_DIM_M / current_max
        if not (0.95 < scale_factor < 1.05):
            for o in vehicle_meshes:
                cur_scale = o.get_scale()
                cur_loc = o.get_location()
                o.set_scale([cur_scale[i] * scale_factor for i in range(3)])
                o.set_location([cur_loc[i] * scale_factor for i in range(3)])
            bpy.context.view_layer.update()
            print(f"[scale] {req.model_path.name}: max_dim "
                  f"{current_max:.2f}m → {TARGET_MAX_DIM_M}m (×{scale_factor:.3f})")

    # Random Z rotation для variety.
    # ВАЖЛИВО: ДОДАТИ Z до існуючого rotation, не replace!
    # glTF спека = Y-up, Blender = Z-up. Importer ставить Euler(π/2, 0, 0) для конверсії.
    # set_rotation_euler([0, 0, z]) обнуляло це → машина лягала на бік (у нативний Y-up).
    rng = np.random.default_rng(req.seed)
    z_rot = float(rng.uniform(0, 2 * math.pi))
    for o in vehicle_meshes:
        cur = o.get_rotation_euler()
        o.set_rotation_euler([cur[0], cur[1], cur[2] + z_rot])
    bpy.context.view_layer.update()

    # Lift vehicle щоб bbox bottom торкався ground plane (z=0).
    # Vehicle origin зазвичай у центрі моделі — без lift половина моделі тоне в землю.
    combined_bbox_post = np.vstack([np.array(o.get_bound_box(local_coords=False))
                                    for o in vehicle_meshes])
    min_z = float(combined_bbox_post[:, 2].min())
    if abs(min_z) > 0.001:
        for o in vehicle_meshes:
            cur_loc = o.get_location()
            o.set_location([cur_loc[0], cur_loc[1], cur_loc[2] - min_z])
        bpy.context.view_layer.update()

    # Центр vehicle після scale+rotation+lift (для камера-target).
    # Використовуємо combined bbox через ВСІ меш-частини, не першу — для multi-mesh
    # GLB (body + wheels окремо) перша мабуть body, її center зміщений.
    combined_bbox_final = np.vstack([np.array(o.get_bound_box(local_coords=False))
                                     for o in vehicle_meshes])
    center = (combined_bbox_final.max(axis=0) + combined_bbox_final.min(axis=0)) / 2

    # 2. Ground plane 10km×10km щоб горизонт не вилазив (drone з 800м бачить ~15км до горизонту).
    ground = bproc.object.create_primitive("PLANE", scale=[5000, 5000, 1])
    ground.set_location([float(center[0]), float(center[1]), 0])
    ground.set_cp("category_id", 0)  # 0 = background, інакше segmentation падає
    # Apply seasonal ground texture (PBR диффузка з Poly Haven)
    if req.ground_texture_path.exists():
        gmat = bproc.material.create_material_from_texture(
            str(req.ground_texture_path), material_name=f"ground_{req.season}"
        )
        ground.replace_materials(gmat)

    # 3. World HDRI sky lighting (strength 2.0 — default 1.0 дає silhouette)
    if req.hdri_path.exists():
        bproc.world.set_world_background_hdr_img(str(req.hdri_path), strength=2.0)

    # 4. Explicit SUN light для daylight. HDRI сам по собі дає тільки ambient sky;
    # без directional sun vehicle виходить майже чорний при overcast HDRI.
    sun = bproc.types.Light()
    sun.set_type("SUN")
    sun.set_energy(5.0)
    # Sun direction: zenith angle 20-50° (~10am-3pm), random azimuth.
    # SUN type у Blender — directional, location ignored, тільки rotation.
    sun_zenith = math.radians(float(rng.uniform(20, 50)))
    sun_azim = float(rng.uniform(0, 2 * math.pi))
    sun.set_rotation_euler([sun_zenith, 0, sun_azim])

    # 5. Drone oblique/nadir геометрія:
    # distance_m = 3D line-of-sight камера→vehicle.
    # H = d·sin(θ), xy = d·cos(θ). cam_z = H > 0 → завжди look-down.
    elev_rad = math.radians(max(req.camera.view_angle_deg, 5.0))
    altitude = req.camera.distance_m * math.sin(elev_rad)
    distance_xy = req.camera.distance_m * math.cos(elev_rad)
    azimuth = float(rng.uniform(0, 2 * math.pi))
    cam_x = float(center[0]) + distance_xy * math.cos(azimuth)
    cam_y = float(center[1]) + distance_xy * math.sin(azimuth)
    cam_z = altitude
    cam_pose = np.array([cam_x, cam_y, cam_z], dtype=float)

    # При view_angle≈90° (nadir) forward vector майже паралельний світовому Z,
    # rotation_from_forward_vec(default up=[0,0,1]) дегенерує (cross product → 0).
    # Special case: будуємо rotation напряму через Euler. Blender camera default
    # дивиться вниз (-Z) при rotation_euler=(0,0,0), top-of-frame = +Y.
    # Z-axis rotation = azimuth для variety орієнтації кадру.
    NADIR_THRESHOLD_DEG = 85.0
    if req.camera.view_angle_deg >= NADIR_THRESHOLD_DEG:
        from mathutils import Euler
        rot_mat = Euler((0.0, 0.0, azimuth), 'XYZ').to_matrix()
        look_at_matrix = bproc.math.build_transformation_mat(cam_pose, rot_mat)
    else:
        look_at_matrix = bproc.math.build_transformation_mat(
            cam_pose,
            bproc.camera.rotation_from_forward_vec(center - cam_pose),
        )
    bproc.camera.add_camera_pose(look_at_matrix)

    # 6. Camera intrinsics
    bproc.camera.set_resolution(req.image_w, req.image_h)
    cam = bpy.context.scene.camera.data
    cam.lens = req.camera.focal_mm
    cam.sensor_width = req.camera.sensor_width_mm

    return look_at_matrix, vehicle_meshes
'''

CAMERA_SAMPLER_PY = r'''"""Stratifier altitude × view-angle.

Будує плоский список (altitude, view_angle) комбінацій з YAML config,
вибирає round-robin або random для кожного кадру.

Pure-Python. bpy не потрібен.
"""

from __future__ import annotations

import random
from dataclasses import dataclass


@dataclass
class CameraSample:
    distance_m: float        # пряма distance camera → vehicle (200-1000м)
    view_angle_deg: float    # елевація camera над горизонтом


def build_grid(
    distances: list[float],
    view_angles: list[float],
) -> list[CameraSample]:
    """Декартів продукт distances × view_angles."""
    return [
        CameraSample(distance_m=d, view_angle_deg=v)
        for d in distances
        for v in view_angles
    ]


def sample_round_robin(
    grid: list[CameraSample],
    n: int,
) -> list[CameraSample]:
    """Рівномірне покриття grid: проходить по колу `n` разів."""
    return [grid[i % len(grid)] for i in range(n)]


def sample_random(
    grid: list[CameraSample],
    n: int,
    seed: int = 0,
) -> list[CameraSample]:
    rng = random.Random(seed)
    return [rng.choice(grid) for _ in range(n)]
'''

SEASON_LIGHTING_PY = r'''"""Season → HDRI + ground texture path resolver.

Не торкається bpy, просто матчить season name до доступних assets.
"""

from __future__ import annotations

import random
from dataclasses import dataclass
from pathlib import Path


@dataclass
class SeasonAssets:
    season: str
    hdri: Path
    ground_texture: Path


def pick_season_assets(
    assets_root: Path,
    season: str,
    seed: int = 0,
) -> SeasonAssets:
    """Випадково (з seed) обирає одну HDRI + одну ground texture для season."""
    hdri_dir = assets_root / "hdri" / season
    ground_dir = assets_root / "textures" / "ground" / season

    hdris = sorted(hdri_dir.glob("*.hdr")) + sorted(hdri_dir.glob("*.exr"))
    grounds = sorted(ground_dir.glob("*_diff_*.png")) + sorted(ground_dir.glob("*_diff_*.jpg"))

    if not hdris:
        raise FileNotFoundError(f"no HDRIs in {hdri_dir}")
    if not grounds:
        raise FileNotFoundError(f"no ground textures in {ground_dir}")

    rng = random.Random(seed)
    return SeasonAssets(
        season=season,
        hdri=rng.choice(hdris),
        ground_texture=rng.choice(grounds),
    )
'''

BBOX_EXTRACTOR_PY = r'''"""COCO → YOLO bbox conversion + min-side filter.

BlenderProc writes COCO JSON; ми конвертуємо у YOLO TXT і фільтруємо bbox < min_side_px.

Pure-Python функції — без bpy/bproc — тестуються локально.
"""

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path


@dataclass
class YoloBox:
    cls: int
    xc: float
    yc: float
    w: float
    h: float
    is_truncated: bool = False
    is_occluded: bool = False

    def to_line(self) -> str:
        return f"{self.cls} {self.xc:.6f} {self.yc:.6f} {self.w:.6f} {self.h:.6f}"


def coco_xywh_to_yolo(
    coco_box: tuple[float, float, float, float],
    image_w: int,
    image_h: int,
) -> tuple[float, float, float, float]:
    """COCO (xmin, ymin, w, h) у pixels → YOLO (xc, yc, w, h) normalized 0..1."""
    x, y, w, h = coco_box
    return (
        (x + w / 2) / image_w,
        (y + h / 2) / image_h,
        w / image_w,
        h / image_h,
    )


def coco_to_yolo(
    coco_json_path: Path,
    image_w: int,
    image_h: int,
    min_side_px: int = 10,
) -> list[tuple[str, list[YoloBox]]]:
    """Читає BlenderProc-written COCO JSON, повертає [(image_filename, [YoloBox, ...]), ...].

    Кадри з усіма bbox < min_side_px стають hard negative (порожній YOLO файл).
    """
    data = json.loads(coco_json_path.read_text(encoding="utf-8"))
    img_by_id = {img["id"]: img for img in data["images"]}
    boxes_by_img: dict[int, list[YoloBox]] = {iid: [] for iid in img_by_id}
    for ann in data.get("annotations", []):
        x, y, w, h = ann["bbox"]
        if min(w, h) < min_side_px:
            continue
        img = img_by_id[ann["image_id"]]
        xc, yc, nw, nh = coco_xywh_to_yolo((x, y, w, h), img["width"], img["height"])
        boxes_by_img[ann["image_id"]].append(
            YoloBox(
                cls=ann["category_id"],
                xc=xc, yc=yc, w=nw, h=nh,
                is_truncated=bool(ann.get("iscrowd", 0)),
            )
        )
    return [
        (img_by_id[iid]["file_name"], boxes)
        for iid, boxes in boxes_by_img.items()
    ]


def write_yolo_label(boxes: list[YoloBox], path: Path) -> None:
    path.write_text("\n".join(b.to_line() for b in boxes) + ("\n" if boxes else ""),
                    encoding="utf-8")


def extract_from_3d_object(
    obj_name: str,
    class_id: int,
    image_w: int,
    image_h: int,
    min_side_px: int = 10,
) -> YoloBox | None:
    """3D bbox projection fallback коли BlenderProc COCO не використовується.

    Lazy import bpy. Беремо 8 вершин bound_box, проєктуємо у camera space,
    обчислюємо axis-aligned 2D bbox. Якщо min-side < min_side_px → None.
    """
    import bpy
    from bpy_extras.object_utils import world_to_camera_view

    scene = bpy.context.scene
    cam = scene.camera
    obj = bpy.data.objects[obj_name]
    coords_2d = []
    for v in obj.bound_box:
        world_co = obj.matrix_world @ __mathutils_vec(v)
        cam_co = world_to_camera_view(scene, cam, world_co)
        coords_2d.append((cam_co.x, 1 - cam_co.y))  # Blender y-up → image y-down
    xs = [c[0] for c in coords_2d]
    ys = [c[1] for c in coords_2d]
    x_min, x_max = max(0, min(xs)), min(1, max(xs))
    y_min, y_max = max(0, min(ys)), min(1, max(ys))
    w, h = x_max - x_min, y_max - y_min
    if w * image_w < min_side_px or h * image_h < min_side_px:
        return None
    return YoloBox(
        cls=class_id,
        xc=x_min + w / 2,
        yc=y_min + h / 2,
        w=w,
        h=h,
    )


def __mathutils_vec(v):
    from mathutils import Vector
    return Vector(v)
'''

FILES = [
    ("render_runner.py", RENDER_RUNNER_PY),
    ("scene_builder.py", SCENE_BUILDER_PY),
    ("camera_sampler.py", CAMERA_SAMPLER_PY),
    ("season_lighting.py", SEASON_LIGHTING_PY),
    ("bbox_extractor.py", BBOX_EXTRACTOR_PY),
]
for name, content in FILES:
    pathlib.Path(f"{ENGINE}/{name}").write_text(content, encoding="utf-8")
pathlib.Path(f"{ENGINE}/__init__.py").write_text("", encoding="utf-8")
print("engine files written:", sorted(pathlib.Path(ENGINE).glob("*.py")))


In [ ]:
# 4. Write config from heredoc -> Drive
import pathlib

V1_LIGHT_VEHICLE_YAML = r'''# Phase 1 smoke — light_vehicle (cls 7).
# Клон v1_apc_reference.yaml для GAZ Tigr / civilian car / mil-pickup.

version: df-v1.0.0-dev
seed: 42

class:
  name: light_vehicle
  id: 7
  models:
    # Файли мають лежати у datasetforge/assets/models/light_vehicle/
    - gaz_tigr.blend
    - civilian_sedan.blend
    - mil_pickup.blend

count: 20             # Phase 1 smoke; Phase 2 = 200; Phase 3 = 600
image_size: [640, 640]

camera:
  # distance_m — 3D line-of-sight камера→техніка.
  # ВАЖЛИВО (tier per-class): light_vehicle ~4м. На distance>500м з hfov=15°
  # це <17 px = шум. 700/1000м тут ВИРІЗАНО — це фізично безглузда комбінація
  # для дрібних класів. Tank/AD/truck (8-12м) можуть тримати весь діапазон у власних
  # configs. Див. labeling_guidelines.md / pixel_budget.md.
  distance_m: [150, 200, 300, 400, 500]
  # view_angle — елевація над горизонтом.
  # 10-30° cruising oblique + 90° nadir для diversity ракурсів (НЕ для видимості).
  view_angle_deg: [10, 15, 20, 25, 30, 90]
  # hfov 15° = drone ISR telephoto (Mavic 3 Enterprise zoom, ScanEagle).
  # УВАГА: це освічене вгадування. Реальна камера дрона не специфікована —
  # коли буде spec, перерахувати lens (zoom-gimbal ~10-15° vs wide ~50-70°).
  hfov_deg: 15
  sensor:
    width: 1920
    height: 1080
    modality: EO

scene:
  landscapes: [field, forest_belt, dirt_road]
  seasons: [summer, autumn_mud, winter, spring]
  weather: [clear, fog, haze, rain]
  time_of_day: [day, dusk]

degradation:
  jpeg_quality: [60, 95]
  motion_blur_px: [0, 8]
  atmosphere_strength: [0.0, 0.4]

hard_negatives:
  ratio: 0.15
  types:
    - empty_landscape

output:
  format: yolo
  split:
    train: 0.7
    valid: 0.2
    test: 0.1
  metadata_sidecar: true
'''

CFG_FILES = [
    ("v1_light_vehicle.yaml", V1_LIGHT_VEHICLE_YAML),
]
for name, content in CFG_FILES:
    pathlib.Path(f"{CONFIGS}/{name}").write_text(content, encoding="utf-8")
print("configs written:", sorted(pathlib.Path(CONFIGS).glob("*.yaml")))


In [ ]:
# 5. Перевір assets (HDRI, ground textures, vehicle models)
import subprocess
for sub in ('hdri', 'textures/ground', 'models/light_vehicle'):
    print(f'--- {sub} ---')
    out = subprocess.run(['ls', '-la', f'{ASSETS}/{sub}'], capture_output=True, text=True)
    print(out.stdout[:1000] or '(empty)')

# Якщо models/light_vehicle порожнє — пiдтягни GAZ Tigr .glb з Drive folder:
#   gdown --folder https://drive.google.com/drive/folders/1WdGkLf9H-FbscBWjkhwSHo1_L40VLqZU -O {ASSETS}/models/light_vehicle/


## G0 BlenderProc quickstart sanity

Monkey-only render, перевіряє що BlenderProc + GPU працюють. Skip якщо вже PASS у попередньому запуску.


In [ ]:
# 6. G0: дефолтна BlenderProc сцена з Suzanne (мавпа) — verify GPU + Cycles.
# MPLBACKEND=agg обовʼязково: Colab default 'module://matplotlib_inline.backend_inline'
# не підтримується bundled matplotlib усередині Blender → crash на import.
# Префікс env у shell — НЕ зачіпає kernel matplotlib (cell 12 inline-display).
!cd /content && MPLBACKEND=agg blenderproc quickstart 2>&1 | tail -20


In [ ]:
# 7. G0 verify: відкрити HDF5 і display
import h5py, numpy as np
from PIL import Image
import io

import glob
h5s = glob.glob('/content/output/*.hdf5')
if h5s:
    with h5py.File(h5s[0], 'r') as f:
        print('keys:', list(f.keys()))
        if 'colors' in f:
            arr = np.array(f['colors'])
            print('colors:', arr.shape, arr.dtype)
            Image.fromarray(arr.astype(np.uint8)).save('/content/g0_monkey.png')
            print('saved /content/g0_monkey.png')
else:
    print('no hdf5 — quickstart може ще не завершився')


## G1 Render light_vehicle smoke (20 кадрів)

`render_runner.py` запускається через `blenderproc run` (НЕ через python — це BlenderProc requirement,
бо bproc init потребує власного Blender process).

Working dir = parent of `datasetforge/` бо `render_runner.py` робить `sys.path.insert(0, parent.parent.parent)`.


In [ ]:
# 8. G1 render — light_vehicle × 20 кадрів
import subprocess, os
# MPLBACKEND=agg тільки для subprocess: Colab default ламає bundled matplotlib у Blender.
# kernel matplotlib залишається inline (потрібно для cell 12 bbox overlay display).
env = {**os.environ, 'MPLBACKEND': 'agg'}
proc = subprocess.run(
    ['blenderproc', 'run', f'{ENGINE}/render_runner.py',
     '--config', f'{CONFIGS}/v1_light_vehicle.yaml',
     '--n', '20',
     '--out', OUT,
     '--assets-root', ASSETS,
     '--seed', '42'],
    cwd=DRIVE,
    env=env,
    capture_output=False,
    text=True,
)
print('exit code:', proc.returncode)


In [ ]:
# 9. G1 verify counts + metadata sanity
import glob, json, pathlib
imgs = sorted(glob.glob(f'{OUT}/images/*.jpg'))
lbls = sorted(glob.glob(f'{OUT}/labels/*.txt'))
metas = sorted(glob.glob(f'{OUT}/metadata/*.json'))
print(f'images: {len(imgs)}, labels: {len(lbls)}, metadata: {len(metas)}')

from PIL import Image
for p in imgs[:5]:
    img = Image.open(p)
    meta_p = p.replace('/images/', '/metadata/').replace('.jpg', '.json')
    meta = json.loads(pathlib.Path(meta_p).read_text())
    print(f'{pathlib.Path(p).name}: {img.size} mode={img.mode}  '
          f'd={meta["distance_m"]:.0f}m alt={meta["altitude_m"]:.0f}m '
          f'angle={meta["view_angle_deg"]:.0f}° n_boxes={meta["n_boxes"]}')


In [ ]:
# 10. G1 visual smoke — display 4 кадри inline
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import glob, pathlib

imgs = sorted(glob.glob(f'{OUT}/images/*.jpg'))[:4]
fig, axes = plt.subplots(1, len(imgs), figsize=(4*len(imgs), 4))
if len(imgs) == 1:
    axes = [axes]
for ax, img_p in zip(axes, imgs):
    img = Image.open(img_p).convert('RGB').copy()
    draw = ImageDraw.Draw(img)
    lbl_p = img_p.replace('/images/', '/labels/').replace('.jpg', '.txt')
    W, H = img.size
    for line in pathlib.Path(lbl_p).read_text().strip().splitlines():
        if not line.strip(): continue
        _cls, xc, yc, w, h = (float(x) for x in line.split())
        x1, y1 = (xc - w/2)*W, (yc - h/2)*H
        x2, y2 = (xc + w/2)*W, (yc + h/2)*H
        draw.rectangle([x1, y1, x2, y2], outline='lime', width=2)
    ax.imshow(img); ax.axis('off')
    ax.set_title(pathlib.Path(img_p).stem, fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
# 11. Pack + download .zip для local review
import shutil
zip_path = '/content/smoke_v1.zip'
shutil.make_archive('/content/smoke_v1', 'zip', OUT)
print(f'created {zip_path} ({pathlib.Path(zip_path).stat().st_size // 1024} KB)')
from google.colab import files
files.download(zip_path)
